# Создавање сопствен сет податоци за процена на водата во кореновата зона

Овој notebook е почетната точка ако имате податоци за пристап до CDSE и сакате да создадете сопствено податочно множество за проектот **„Процена на водата во кореновата зона со сателитски податоци“**.

Постапката има два чекори:

1. Го задавате периодот, филтерот и папката за податоците, со патека во однос на главната проектна папка. Потоа создавате нова датотека со поставки.
2. Го овозможувате преземањето на метеоролошките податоци од Open-Meteo и обработените податоци од Sentinel-2 L2A за истиот период и област.

При преземањето се зачувуваат изворните одговори од сервисите, ознаките за квалитет, податоците за снимките од каталогот, координатите на парцелата што ја истражувате, поставките и контролните суми на датотеките.

Главниот notebook за процена потоа може да ги користи зачуваните податоци **без интернет и без податоци за најава**. Постојните сетови податоци не се препишуваат.

## Подготовка пред преземањето

Создајте OAuth клиент во CDSE и зачувајте ги добиените две вредности во приватна датотека за променливи на околината, на пример `.env`. Во неа треба да бидат зададени `SENTINEL_CLIENT_ID` и `SENTINEL_CLIENT_SECRET`. Не ги внесувајте нивните вредности во notebook-от и не ја додавајте приватната датотека во Git.

**Почетниот и крајниот датум се вклучени во анализата.** При барањето податоци, како крај на интервалот се испраќа почетокот на денот по `end_date`. Така се опфаќаат цели денови според UTC. За период од 90 дена, последниот вклучен датум е 89 дена по почетниот.

Зададената граница на парцелата, изборот на пченица и отсуството на наводнување се претпоставки за наставниот пример. Тие остануваат претпоставки сè додека не ги замените со проверени податоци.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from root_zone_water.config import generate_config

# Пред преземањето, задајте ги периодот, филтерот и патеките подолу.
START_DATE = "2025-05-01"
END_DATE = "2025-07-29"
FILTER_NAME = "ekf"                 # "ekf", "ukf" или "open_loop"
DATASET_PATH = "data/my-root-zone-dataset"
CONFIG_PATH = "config_my_dataset.json"
ENV_FILE = ".env"                   # приватна датотека со податоците за пристап до CDSE

custom_config = generate_config(
    start_date=START_DATE,
    end_date=END_DATE,
    filter_name=FILTER_NAME,
    dataset_path=DATASET_PATH,
    output_path=ROOT / CONFIG_PATH,
)
display({"Датотека со поставки": CONFIG_PATH,
         "Папка со податоци": custom_config["cache_dir"],
         "Почетен датум": custom_config["start_date"],
         "Краен датум": custom_config["end_date"],
         "Избран филтер": custom_config["filter"],
         "Режим на работа": custom_config["mode"]})

## Преземање и зачувување на податоците

Следната ќелија испраќа барања преку интернет и го користи дозволениот обем на пристап до сервисот CDSE. Затоа преземањето првично е исклучено.

Додека ги уредувате или проверувате поставките, оставете `RUN_DOWNLOAD=False`. Кога датотеката со податоците за пристап е подготвена и поставките се точни, променете ја вредноста во `True` и извршете ја ќелијата.

Новата датотека со поставки веќе ја содржи патеката до папката за податоците. По преземањето можете да ја користите за извршување на процената, како што е објаснето подолу.

In [ ]:
RUN_DOWNLOAD = False

if RUN_DOWNLOAD:
    from dotenv import load_dotenv
    from root_zone_water.acquisition import acquire

    load_dotenv(ROOT / ENV_FILE, override=False)
    manifest = acquire(custom_config, ROOT / DATASET_PATH)
    display({"Состојба на преземањето": manifest["status"],
             "Денови со метеоролошки податоци": manifest.get("weather_days"),
             "Број на сателитски записи": manifest.get("satellite_rows"),
             "Денови со прифатен NDMI": manifest.get("usable_ndmi_days"),
             "Пријавени грешки": manifest.get("errors")})
else:
    print("Преземањето е исклучено. Поставете RUN_DOWNLOAD=True откако ќе ја подготвите локалната датотека со податоците за пристап.")

## Користење на зачуваните податоци без интернет

Пред да ја извршите процената, проверете ја датотеката `data/my-root-zone-dataset/manifest.json`. Ако сте избрале друга папка, отворете ја `manifest.json` во таа папка.

Во неа се запишани покриеноста со метеоролошки податоци, прифатените и отфрлените сателитски интервали, бројот на примероци, снимките пронајдени во каталогот, испратените барања, изворните одговори и поставките за обработка.

За извршување од командната линија, наведете ја создадената датотека со поставки:

```powershell
python -m root_zone_water.runner --config config_my_dataset.json --output outputs/my-dataset
```

Ако сакате да ги користите податоците во главниот notebook за процена, пренесете ги `cache_dir`, датумите и избраниот филтер од `config_my_dataset.json` во `config.json`. Потоа извршете:

```powershell
python execute_notebook.py
```

Режимот `cached` ги чита зачуваните податоци. Не ги вчитува податоците за пристап и не испраќа API барања.

## Кој филтер го избираме?

- **`ekf`** го користи аналитичкиот извод на водниот биланс по состојбата, односно Јакобијанот по количеството вода.
- **`ukf`** пропушта скаларни сигма-точки низ истиот воден биланс и од нив ги добива предвидената средна вредност и коваријанса.
- **`open_loop`** предвидува само со моделот. Процената не се корегира со NDMI.

Сите три начини ги користат истите влезни податоци, истата врска меѓу NDMI и водата, истите правила за прифаќање на податоците и истите поставки за неизвесноста.

Проценетото количество вода зависи од зададените претпоставки. Sentinel-2 дава индиректни информации за вегетацијата, а не теренски мерења на водата околу корените. За деновите без прифатена снимка не се додаваат интерполирани или измислени набљудувања.